# Assignment 6: AI-Based Style Transfer

In this notebook I take one photo and run it through two style transfer methods:

1. **CycleGAN** (pretrained, photo -> Van Gogh style)
2. **Neural Style Transfer** with VGG19 (using Van Gogh's *Starry Night* as the style image)

I picked Van Gogh for both so the comparison is fair (same artist, different method).

**Note:** run this on Google Colab with GPU on (Runtime > Change runtime type > T4 GPU).

## 1. Setup

In [ ]:
import os, sys, glob, shutil, requests
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load the content image

Upload your own photo (a landscape or building works best). If you skip the upload, it will use a sample photo from the internet instead.

In [ ]:
# image size: both methods will use the same size
# (width and height are multiples of 4 because CycleGAN needs that)
W, H = 512, 384

os.makedirs("input_dir", exist_ok=True)
content_path = "input_dir/content.jpg"

try:
    from google.colab import files
    print("Upload your photo (or cancel to use the sample):")
    uploaded = files.upload()
    if uploaded:
        name = list(uploaded.keys())[0]
        Image.open(name).convert("RGB").resize((W, H)).save(content_path)
except Exception as e:
    print("No upload, will use sample image.", e)

if not os.path.exists(content_path):
    # sample photo (change the URL if it ever stops working)
    url = "https://pytorch.org/tutorials/_static/img/neural-style/dancing.jpg"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    Image.open(BytesIO(r.content)).convert("RGB").resize((W, H)).save(content_path)

content_img = Image.open(content_path).convert("RGB")
plt.imshow(content_img); plt.axis("off"); plt.title("Original image"); plt.show()

## 3. Method 1: CycleGAN (pretrained)

I'm using the official `pytorch-CycleGAN-and-pix2pix` repo, which has pretrained models. The `style_vangogh` model turns a photo into a Van Gogh style painting.

**Cycle consistency (the main idea behind CycleGAN):**
CycleGAN learns from two unpaired sets of images (photos and Van Gogh paintings), so there is no "photo + matching painting" pair to learn from. It has two generators: G (photo -> painting) and F (painting -> photo). The cycle consistency loss says that if you translate a photo to a painting and then back again, you should get the original photo back, i.e. `F(G(x)) ≈ x`. This forces the generator to keep the structure of the photo (buildings, trees, horizon) and only change the style, since otherwise it couldn't get back to the original.

In [ ]:
!git clone -q https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix
%cd pytorch-CycleGAN-and-pix2pix
!pip install -q dominate
!bash ./scripts/download_cyclegan_model.sh style_vangogh

In [ ]:
# copy our image into the folder the repo reads from
os.makedirs("my_input", exist_ok=True)
shutil.copy("../input_dir/content.jpg", "my_input/content.jpg")

gpu = "0" if torch.cuda.is_available() else "-1"
!python test.py --dataroot ./my_input --name style_vangogh_pretrained --model test --no_dropout --dataset_mode single --preprocess none --gpu_ids {gpu} --num_test 1

In [ ]:
out_file = glob.glob("results/style_vangogh_pretrained/test_latest/images/*fake*.png")[0]
cyclegan_img = Image.open(out_file).convert("RGB")
print(out_file, cyclegan_img.size)
plt.imshow(cyclegan_img); plt.axis("off"); plt.title("CycleGAN output"); plt.show()
%cd ..

## 4. Method 2: Neural Style Transfer (VGG19)

This is the Gatys et al. method. There is no training of a generator here. We start from a copy of the content image and change its *pixels* directly with an optimizer so that:

- **content loss**: its VGG features stay close to the content image's features
- **style loss**: the Gram matrices of its features match the style painting's (Gram matrix = which textures/colours appear together)

In [ ]:
# style painting: Starry Night by Van Gogh
style_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg/1280px-Van_Gogh_-_Starry_Night_-_Google_Art_Project.jpg"
r = requests.get(style_url, headers={"User-Agent": "Mozilla/5.0"})
style_pil = Image.open(BytesIO(r.content)).convert("RGB").resize((W, H))
plt.imshow(style_pil); plt.axis("off"); plt.title("Style image (Starry Night)"); plt.show()

In [ ]:
to_tensor = transforms.ToTensor()
content = to_tensor(content_img).unsqueeze(0).to(device)
style = to_tensor(style_pil).unsqueeze(0).to(device)

# VGG19 (only the conv part, frozen)
vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()
for p in vgg.parameters():
    p.requires_grad_(False)
# inplace ReLU would overwrite the conv outputs we save, so turn it off
for layer in vgg:
    if isinstance(layer, nn.ReLU):
        layer.inplace = False

mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

# layers (names from the pytorch tutorial)
content_layers = ["conv4_2"]
style_layers = ["conv1_1", "conv2_1", "conv3_1", "conv4_1", "conv5_1"]

def get_features(x):
    x = (x - mean) / std
    feats = {}
    block, i = 1, 0
    for layer in vgg:
        x = layer(x)
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f"conv{block}_{i}"
        elif isinstance(layer, nn.ReLU):
            pass
        elif isinstance(layer, nn.MaxPool2d):
            block += 1; i = 0
        if isinstance(layer, nn.Conv2d) and (name in content_layers or name in style_layers):
            feats[name] = x
    return feats

def gram(x):
    b, c, h, w = x.size()
    f = x.view(c, h * w)
    return f @ f.t() / (c * h * w)

content_feats = get_features(content)
style_grams = {l: gram(f) for l, f in get_features(style).items() if l in style_layers}

In [ ]:
# start from the content image and optimise its pixels
target = content.clone().requires_grad_(True)
optimizer = optim.LBFGS([target])

style_weight = 1e6
content_weight = 1
steps = 300
run = [0]

while run[0] <= steps:
    def closure():
        with torch.no_grad():
            target.clamp_(0, 1)
        optimizer.zero_grad()
        feats = get_features(target)

        c_loss = F.mse_loss(feats["conv4_2"], content_feats["conv4_2"])
        s_loss = 0
        for l in style_layers:
            s_loss += F.mse_loss(gram(feats[l]), style_grams[l])

        loss = content_weight * c_loss + style_weight * s_loss
        loss.backward()

        run[0] += 1
        if run[0] % 50 == 0:
            print(f"step {run[0]}: content {c_loss.item():.4f}, style {s_loss.item():.6f}")
        return loss
    optimizer.step(closure)

with torch.no_grad():
    target.clamp_(0, 1)
nst_img = transforms.ToPILImage()(target.squeeze(0).cpu())
plt.imshow(nst_img); plt.axis("off"); plt.title("NST output"); plt.show()

## 5. Side by side comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, im, t in zip(axes, [content_img, cyclegan_img, nst_img],
                     ["Original", "CycleGAN (Van Gogh)", "Neural Style Transfer (Starry Night)"]):
    ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.tight_layout()
plt.savefig("comparison.png", dpi=150)
plt.show()

In [ ]:
# optional: download the images for the submission document
try:
    from google.colab import files
    files.download("comparison.png")
except Exception:
    pass

## 6. Written comparison

*(Draft based on what these two methods usually produce. Look at your own output above and change the wording if your result looks different.)*

- **Content preservation:** CycleGAN kept the original content better. Shapes and layout stayed almost the same, and the cycle consistency loss is the reason, because the generator has to keep enough structure to be able to map back to the original photo. NST also kept the main outline, but fine details got distorted.
- **Stronger style transformation:** NST gave the stronger style change. It copied the thick swirly brush strokes and colours of Starry Night much more obviously, while CycleGAN mostly changed the colours and gave a light painterly texture.
- **Other differences:** CycleGAN is learned once and then runs in a second (one forward pass), but it is limited to the styles it was trained on. NST takes a few minutes per image because it optimizes the pixels, but it can use any painting as the style.